In [1]:
import os
import io
import sys
from dotenv import load_dotenv
import gradio as gr
import subprocess
from IPython.display import Image, display, Markdown

In [2]:
load_dotenv(override=True)

True

In [3]:
openai_api_key = os.getenv("OPENAI_API_KEY")

In [4]:
from openai import OpenAI

openai = OpenAI(api_key=openai_api_key)

In [5]:
OPENAI_MODEL = "gpt-5-nano"

In [6]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()

In [8]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide with the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute this code:
```python
compile_command = # something here
compile_result = subprocess.run(compile_command, capture_output=True, text=True, check=True)
run_command = # something here
run_result = subprocess.run(run_command, capture_output=True, text=True, check=True)
return run_result.stdout
```

Please tell me exactly what I should use for the compile_command and run_command. 

System Info:
{system_info}   

"""

In [9]:
response = openai.chat.completions.create(model=OPENAI_MODEL,
messages=[
    {"role":"user", "content":message}
])
display(Markdown(response.choices[0].message.content))

Short answer:
- Yes. You currently have no C++ compiler installed (gcc, g++, clang, and msvc_cl are empty).
- The simplest way to get started on Windows is to install Visual Studio Build Tools (MSVC) and then compile with cl. I’ll also give a lighter alternative with MinGW-w64 if you prefer.

Option A: Visual Studio Build Tools (MSVC) – the simplest single-file workflow
Why this is simplest: a straightforward compiler with good Windows integration, and you don’t need to manage a separate toolchain.

Steps to install
1) Open PowerShell or Windows Terminal as administrator.
2) Install the Visual Studio Build Tools (just the C++ toolset):
   - Command:
     winget install -e --id Microsoft.VisualStudio.2022.BuildTools
   - If you have any prompts, approve them.
3) After install, use the Developer Command Prompt for VS, or ensure the VC environment is loaded:
   - You can search for “Developer Command Prompt for VS 2022” and open it, or run vcvars64.bat from the build tools to set up the environment.
4) Put main.cpp in a folder, and open a terminal in that folder.
5) Compile:
   - Basic: cl /EHsc main.cpp
   - Or explicitly specify output: cl /EHsc /Fe:main.exe main.cpp
6) Run:
   - main.exe

If you want to run this from Python, use:
- compile_command = ["cl", "/EHsc", "main.cpp", "/Fe:main.exe"]
- run_command = ["main.exe"]

Notes:
- The cl compiler may require you to run from a VS Developer Command Prompt to have all environment variables set. If you’re not using that, you can run vcvars64.bat (found in the Visual Studio Build Tools folder) to set the environment first.

Option B: MinGW-w64 via MSYS2 (lighter, but a bit more setup)
If you prefer a GCC-based toolchain, this is a common alternative.

Steps to install
1) Download and install MSYS2 from https://www.msys2.org
2) Open the MSYS2 MinGW 64-bit shell (not the 32-bit one) and update:
   - pacman -Syu
   - Close and reopen the shell, then run:
     pacman -Syu
3) Install the GCC toolchain:
   - pacman -S mingw-w64-x86_64-gcc
4) Add MSYS2’s mingw64 bin to your PATH (example path: C:\msys64\mingw64\bin)
5) In a normal Windows Command Prompt or PowerShell, navigate to your folder with main.cpp and compile:
   - g++ -std=c++17 main.cpp -o main.exe
6) Run:
   - main.exe

If you want to run this from Python, use:
- compile_command = ["g++", "-std=c++17", "main.cpp", "-o", "main.exe"]
- run_command = ["main.exe"]

What I can tailor for you
- Tell me which path you prefer (MSVC Build Tools vs. MinGW-w64), and I’ll give you the exact, copy-paste commands using your actual folder paths.
- If you want, I can give you a single Python snippet that auto-detects the toolchain and runs the compile and execute steps accordingly.

In [10]:
compile_command = ["g++", "-std=c++17", "main.cpp", "-o", "main.exe"]
run_command = ["main.exe"]

In [11]:
system_prompt = f"""
Your task is to convert a Python code into high performance C++ code. 
Respond only with the C++ code.
Do not provide any explanation other than the occassional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

In [12]:
def user_prompt_for(python):
    return f"""
    Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
    The system information is:
    {system_info}
    Your response will be written to a file called main.cpp and then compiled and executed. 
    The compilation command is: {compile_command}
    Respond only with the C++ code.
    Python code to port:
    ```python
    {python}
    ```
    """

In [14]:
def messages_for(python):
    return [
        {"role":"system", "content":system_prompt},
        {"role":"user", "content":user_prompt_for(python)}
    ]

In [15]:
def write_output(cpp):
    with open("main.cpp", 'w', encoding='utf-8') as file:
        file.write(cpp)

In [18]:
def port(python):
    reasoning='high'
    response = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for(python), reasoning_effort=reasoning)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)


In [ ]:
pi = """
a=6
b=7
print(a-b)
"""

In [21]:
port(pi)

In [24]:
pi2 = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i*param1 - param2
        result -= (i/j)
        j = i*param1 + param2
        result += (i/j)
    return result

start_time = time.time()
result = calculate(1000, 4,1)*4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution time: {(end_time-start_time):.6f} seconds")
"""

In [25]:
port(pi2)

In [ ]:
def run_python(code):
    globals = {"__builtins__":__builtins__}
    exec(code, globals)